# 05 — Gold dimensions

Build reporting dimensions and bridges only from available Silver entities. The notebook fails before writing if an expected source or column is absent. It does not fabricate source-system closure, reopen, or status-history data.


In [ ]:
AS_OF_DATE = ""  # Optional YYYY-MM-DD, passed by archive replay.
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
CFG_NOTEBOOK_NAME = "00_setup_cfg"
AUDIT_TABLE = "monitoring.cfg_silver_export_load"
TIME_PARSER_POLICY = "CORRECTED"
JOB_RUN_ID = ""  # Parent orchestration correlation ID.
NOTEBOOK_TIMEOUT_SECONDS = 1800


In [ ]:
from datetime import datetime

from notebookutils import mssparkutils
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cfg_result = mssparkutils.notebook.run(
    CFG_NOTEBOOK_NAME,
    NOTEBOOK_TIMEOUT_SECONDS,
    {"AUDIT_TABLE": AUDIT_TABLE, "TIME_PARSER_POLICY": TIME_PARSER_POLICY},
)
print(f"Configuration setup completed: {cfg_result}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
RUN_STARTED_AT = datetime.utcnow()


In [ ]:
def require_columns(source_table, required_columns):
    if not spark.catalog.tableExists(source_table):
        raise ValueError(f"Required Silver source is missing: {source_table}")
    actual = {field.name.lower() for field in spark.table(source_table).schema.fields}
    missing = sorted(set(required_columns) - actual)
    if missing:
        raise ValueError(f"{source_table} is missing required columns: {missing}")


def latest_dimension(source_table, target_table, key_columns, select_columns):
    """Write one current row per natural key from an available Silver source."""
    require_columns(source_table, key_columns + [source for source, _ in select_columns])
    frame = spark.table(source_table)
    order_columns = [
        F.col("export_date").cast("timestamp").desc_nulls_last(),
        F.col("_silver_load_ts").cast("timestamp").desc_nulls_last(),
    ]
    current = (
        frame.withColumn(
            "_gold_dimension_rank",
            F.row_number().over(Window.partitionBy(*key_columns).orderBy(*order_columns)),
        )
        .where(F.col("_gold_dimension_rank") == 1)
        .drop("_gold_dimension_rank")
    )
    dimension = current.select(*[
        F.col(source).alias(target) for source, target in select_columns
    ])
    (dimension.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(target_table))
    print(f"{target_table}: {dimension.count():,} rows from {source_table}")


def copy_bridge(source_table, target_table, required_columns, select_columns):
    require_columns(source_table, required_columns)
    bridge = spark.table(source_table).select(*[
        F.col(source).alias(target) for source, target in select_columns
    ]).dropDuplicates()
    (bridge.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(target_table))
    print(f"{target_table}: {bridge.count():,} rows from {source_table}")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_SCHEMA}.dim_date AS
SELECT
  date_value AS Date,
  YEAR(date_value) AS CalendarYear,
  QUARTER(date_value) AS CalendarQuarter,
  MONTH(date_value) AS CalendarMonthNumber,
  DATE_FORMAT(date_value, 'MMMM') AS CalendarMonthName,
  DATE_FORMAT(date_value, 'yyyy-MM') AS YearMonth,
  DAYOFWEEK(date_value) AS DayOfWeekNumber,
  DATE_FORMAT(date_value, 'EEEE') AS DayOfWeekName,
  DAYOFMONTH(date_value) AS DayOfMonth,
  CASE WHEN DAYOFWEEK(date_value) IN (1, 7) THEN false ELSE true END AS IsWeekday,
  CURRENT_TIMESTAMP() AS GoldModelledAt
FROM (
  SELECT EXPLODE(SEQUENCE(DATE '2020-01-01', DATE '2035-12-31', INTERVAL 1 DAY)) AS date_value
)
""")

latest_dimension(
    "silver.holding_company", "gold.dim_holding_company", ["holding_company_id"],
    [("holding_company_id", "HoldingCompanyID"), ("company_name", "CompanyName"),
     ("town_city", "TownCity"), ("county", "County"), ("postcode", "Postcode"),
     ("country", "Country"), ("export_date", "SourceExportDate")],
)
latest_dimension(
    "silver.provider", "gold.dim_provider", ["provider_id"],
    [("provider_id", "ProviderID"), ("holding_company_id", "HoldingCompanyID"),
     ("provider_name", "ProviderName"), ("provider_status", "ProviderStatus"),
     ("town_city", "TownCity"), ("county", "County"), ("postcode", "Postcode"),
     ("country", "Country"), ("qa_flag", "QAFlag"),
     ("export_date", "SourceExportDate")],
)
latest_dimension(
    "silver.provider_home", "gold.dim_provider_home", ["provider_home_id"],
    [("provider_home_id", "ProviderHomeID"), ("provider_id", "ProviderID"),
     ("service_type", "ServiceType"), ("home_name", "HomeName"),
     ("town_city", "TownCity"), ("county", "County"), ("postcode", "Postcode"),
     ("number_of_registered_beds", "RegisteredBeds"), ("is_spot", "IsSpot"),
     ("qa_flag", "QAFlag"), ("export_date", "SourceExportDate")],
)
latest_dimension(
    "silver.framework", "gold.dim_framework", ["framework_code"],
    [("framework_code", "FrameworkCode"), ("framework_name", "FrameworkName"),
     ("placement_type", "PlacementType"), ("start_date", "StartDate"),
     ("end_date", "EndDate"), ("export_date", "SourceExportDate")],
)
latest_dimension(
    "silver.framework_category", "gold.dim_framework_category", ["framework_category_id"],
    [("framework_category_id", "FrameworkCategoryID"), ("framework_code", "FrameworkCode"),
     ("category_name", "CategoryName"), ("export_date", "SourceExportDate")],
)


In [ ]:
copy_bridge(
    "silver.provider_framework", "gold.bridge_provider_framework",
    ["provider_framework_id", "provider_id", "framework_code", "qa_flag", "export_date"],
    [("provider_framework_id", "ProviderFrameworkID"), ("provider_id", "ProviderID"),
     ("framework_code", "FrameworkCode"), ("qa_flag", "QAFlag"),
     ("export_date", "SourceExportDate")],
)
copy_bridge(
    "silver.provider_sic_codes", "gold.bridge_provider_sic_code",
    ["provider_id", "sic_code", "export_date"],
    [("provider_id", "ProviderID"), ("sic_code", "SICCode"),
     ("export_date", "SourceExportDate")],
)
latest_dimension(
    "silver.provider_submission_docs", "gold.dim_provider_submission_document", ["document_id"],
    [("document_id", "DocumentID"), ("submission_id", "SubmissionID"),
     ("s3_file_metadata_id", "S3FileMetadataID"), ("document_name", "DocumentName"),
     ("document_type", "DocumentType"), ("expiry_date", "ExpiryDate"),
     ("last_updated", "LastUpdated"), ("next_review_date", "NextReviewDate"),
     ("service_type", "ServiceType"), ("home_id", "HomeID"),
     ("start_date", "StartDate"), ("export_date", "SourceExportDate")],
)

require_columns("silver.referral", ["placement_type", "referral_status"])
require_columns("silver.ipa", ["placement_type"])
placement_types = (
    spark.table("silver.referral").select(F.col("placement_type").alias("PlacementType"))
    .unionByName(spark.table("silver.ipa").select(F.col("placement_type").alias("PlacementType")))
    .where(F.col("PlacementType").isNotNull() & (F.trim(F.col("PlacementType")) != ""))
    .dropDuplicates()
    .withColumn("GoldModelledAt", F.current_timestamp())
)
(placement_types.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("gold.dim_placement_type"))

referral_statuses = (
    spark.table("silver.referral").select(F.col("referral_status").alias("ReferralStatus"))
    .where(F.col("ReferralStatus").isNotNull() & (F.trim(F.col("ReferralStatus")) != ""))
    .dropDuplicates()
    .withColumn("GoldModelledAt", F.current_timestamp())
)
(referral_statuses.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("gold.dim_referral_status"))
print(f"Gold dimensions completed; AS_OF_DATE={AS_OF_DATE or 'latest'}; started={RUN_STARTED_AT.isoformat()}")
